In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Multi-Paper Research QA — Source-Backed Answers Across Papers

## 1. Project Overview

**Task:** Index multiple research papers, retrieve evidence from more than one paper, and answer research questions with explicit source backing.

**Approach:** Build a small paper corpus with section-aware metadata, chunk it carefully, embed it into ChromaDB, retrieve with paper-diversity constraints, and synthesize answers with inline citations. For harder questions, add **multi-hop retrieval**: decompose the question into smaller hops, retrieve each hop separately, then combine the evidence.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for decomposition and synthesis
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — local embeddings
- `ChromaDB` — vector store
- `LangChain` — text splitting and message formatting

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Index multiple papers** with title, year, venue, and section metadata |
| 2 | **Retrieve evidence from more than one paper** instead of collapsing onto a single source |
| 3 | **Explain multi-hop retrieval** and when it helps literature synthesis |
| 4 | **Generate source-backed answers** with explicit paper-and-section citations |
| 5 | **Measure coverage** by checking how many distinct papers support an answer |
| 6 | **Compare one-hop and multi-hop retrieval** on harder research questions |

## 3. Problem Statement

Many research questions are not answered by a single paper. Examples:
- *"How do section-aware chunking and query decomposition complement each other in scientific QA?"*
- *"Which retrieval strategy improves evidence coverage, and what later validation step makes the answer more trustworthy?"*
- *"Why does verification work better when retrieval already covers multiple papers?"*

A naive RAG system often retrieves only one paper very well and misses the second paper needed for a proper synthesis.

## 4. Why This Matters

- **Literature review assistants** need cross-paper synthesis, not just single-document lookup
- **Source diversity** matters because one paper rarely covers the full answer surface
- **Multi-hop retrieval** matters because some questions contain two linked information needs
- **Evidence-backed answers** are more trustworthy when they cite multiple papers explicitly

## 5. Pipeline Architecture

```
Research Papers
     │
     ├── Parse into sections (Abstract, Method, Results, Limitations)
     ├── Chunk with paper + section metadata
     ├── Embed and index in ChromaDB
     ├── Retrieve with paper-diversity constraints
     ├── Optional: multi-hop question decomposition
     │      ├── Hop 1 retrieval
     │      ├── Hop 2 retrieval
     │      └── Merge evidence across papers
     └── Generate answer with inline citations and confidence
```

### One-Hop vs Multi-Hop Retrieval

| Mode | What It Does | Best For | Failure Mode |
|------|---------------|----------|--------------|
| **One-hop retrieval** | Single semantic search using the full question | Direct questions answered by one or two obvious chunks | Misses linked sub-questions hidden inside a complex prompt |
| **Multi-hop retrieval** | Breaks a question into smaller retrieval hops, gathers evidence per hop, then synthesizes | Cross-paper synthesis and compare-and-connect questions | More latency and orchestration complexity |

## 6. Environment Setup

This notebook uses the same local stack as the other NLP RAG projects in this repository.

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface langchain-text-splitters chromadb sentence-transformers

print("Dependencies: langchain, langchain-ollama, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

We keep temperatures low because research QA should favor grounded synthesis over stylistic creativity.

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter
from typing import Optional

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 6
MIN_PAPERS = 2
TEMP_ANSWER = 0.1
TEMP_PLAN = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Chunking: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"Retrieval: top-{TOP_K}, minimum papers={MIN_PAPERS}")

## 9. Helper Functions

These are the core notebook utilities: clean LLM output, parse JSON safely, wrap long output, and call the local model.

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


def short_citation(meta: dict) -> str:
    return f"{meta['title']} ({meta['year']}) | {meta['section']}"

## 10. Paper Corpus

To keep the notebook runnable and self-contained, we use **synthetic but realistic paper excerpts**.

## 11. Inspect the Corpus

We will index three papers that complement each other rather than duplicate one another.

In [ ]:
PAPERS = [
    {
        "paper_id": "p1",
        "title": "Section-Aware Chunking Improves Scientific Retrieval",
        "year": 2024,
        "venue": "ACL Findings",
        "topic": "retrieval",
        "authors": ["Mina Patel", "Evan Ross"],
        "content": """# Section-Aware Chunking Improves Scientific Retrieval

## Abstract
Scientific papers contain highly structured sections, but many retrieval systems chunk them as flat text. We test whether respecting section boundaries improves literature question answering. On a 3,200-question benchmark, section-aware chunking improves Recall@5 from 61.4 to 73.8 and citation localization from 58.0 to 82.1.

## Introduction
Research questions often target specific sections such as methods, limitations, or results. When a chunk mixes unrelated sections, embeddings blur the signal. Our hypothesis is that section labels act as retrieval priors and make evidence easier to localize.

## Method
We parse headings, keep each section intact when possible, and add metadata for section type and paper title. We compare this against uniform fixed-size chunking with the same average token budget.

## Results
Section-aware chunking improves evidence coverage for synthesis tasks because the retriever surfaces cleaner, more semantically coherent evidence. Human annotators reported that section-aware chunks were easier to cite and combine across papers.

## Limitations
Section-aware chunking is only as good as the section parser. Papers with poor PDF structure or missing headings still require fallback heuristics."""
    },
    {
        "paper_id": "p2",
        "title": "Query Decomposition for Multi-Paper Scientific Question Answering",
        "year": 2025,
        "venue": "EMNLP",
        "topic": "multi_hop_retrieval",
        "authors": ["Leah Gomez", "Noah Chen"],
        "content": """# Query Decomposition for Multi-Paper Scientific Question Answering

## Abstract
Complex literature questions often contain multiple linked sub-questions. We decompose each query into retrieval hops before synthesis. On a scientific synthesis benchmark, decomposition increases the average number of supporting papers per answer from 1.4 to 2.7 and raises answer quality from 64.2 to 78.9.

## Motivation
A question like 'Which retrieval change improves coverage, and what evaluation step makes the final synthesis trustworthy?' asks for two facts that may live in different papers. A single dense retrieval call often over-focuses on one paper.

## Method
We prompt an LLM planner to break a question into 2 to 3 sub-questions. Each sub-question is retrieved independently, then evidence is deduplicated and recombined for the final synthesis step.

## Results
Multi-hop retrieval improves coverage on compare-and-connect questions, especially when the final answer must name a method from one paper and a validation procedure from another. The largest gain appears when combined with paper-diverse reranking.

## Failure Analysis
Poor decomposition can introduce drift if a sub-question becomes more general than the original question. Retrieval diversity helps, but the planner still needs tight prompts."""
    },
    {
        "paper_id": "p3",
        "title": "Evidence Calibration and Citation Verification in Literature QA",
        "year": 2025,
        "venue": "NAACL",
        "topic": "citation_verification",
        "authors": ["Priya Nandakumar", "Omar Lee"],
        "content": """# Evidence Calibration and Citation Verification in Literature QA

## Abstract
Literature assistants often produce fluent answers with weak evidence. We introduce a citation verification pass that checks whether each answer claim is directly supported by retrieved passages. Unsupported claims fall from 18.6% to 7.1%, while human trust rises from 3.2 to 4.4 out of 5.

## Background
Cross-paper synthesis is useful only if each claim is anchored to a source. Requiring evidence from at least two papers improves robustness, but the synthesis step still needs a downstream verification check.

## Method
After answer generation, a verifier re-reads the answer and the evidence set, then flags unsupported statements, missing citations, and over-generalized claims. We also record how many distinct papers support the final answer.

## Results
Verification works best when retrieval already covers multiple papers. When the retriever returns a narrow evidence set from a single paper, verification mostly catches unsupported claims after the fact instead of preventing them.

## Limitations
Citation verification does not fix missing evidence. It improves faithfulness, but cannot replace good retrieval or question decomposition."""
    }
]

print(f"Paper count: {len(PAPERS)}")
for paper in PAPERS:
    sections = re.findall(r"^##\s+(.+)", paper["content"], flags=re.MULTILINE)
    print(f"- {paper['paper_id']}: {paper['title']} ({paper['year']}, {paper['venue']})")
    print(f"  Topic: {paper['topic']}, sections: {sections}")

## 12. Parse Paper Sections

Each paper section becomes a first-class retrieval unit with metadata.

## 13. Why Section-Aware Chunking Helps

A literature question like *"What improves evidence coverage, and what verification step makes the result safer?"* usually maps to different parts of different papers:
- coverage improvements may be described in **Results**
- implementation details often live in **Method**
- caveats often live in **Limitations**

## 14. Create Chunks

We split long sections while preserving paper identity and section labels.

In [ ]:
def parse_paper_sections(paper: dict) -> list[dict]:
    sections = []
    parts = re.split(r"(?=^##\s+)", paper["content"], flags=re.MULTILINE)
    for part in parts:
        part = part.strip()
        if not part.startswith("## "):
            continue
        match = re.match(r"^##\s+(.+)", part)
        section_name = match.group(1).strip() if match else "Unknown"
        sections.append({
            "text": part,
            "metadata": {
                "paper_id": paper["paper_id"],
                "title": paper["title"],
                "year": str(paper["year"]),
                "venue": paper["venue"],
                "topic": paper["topic"],
                "section": section_name,
            },
        })
    return sections


section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

all_sections = []
all_chunks = []
for paper in PAPERS:
    paper_sections = parse_paper_sections(paper)
    all_sections.extend(paper_sections)
    for section in paper_sections:
        parts = section_splitter.split_text(section["text"]) if len(section["text"]) > CHUNK_SIZE else [section["text"]]
        for i, part in enumerate(parts):
            all_chunks.append({
                "text": part,
                "metadata": {
                    **section["metadata"],
                    "chunk_index": i,
                    "chunk_total": len(parts),
                },
            })

print(f"Section count: {len(all_sections)}")
print(f"Chunk count: {len(all_chunks)}")
print("Chunks per paper:")
for paper_id, count in Counter(chunk["metadata"]["paper_id"] for chunk in all_chunks).items():
    print(f"  {paper_id}: {count}")

print("\nExample chunk:")
example = all_chunks[0]
print(short_citation(example["metadata"]))
print(example["text"][:350] + "...")

## 15. Build the Vector Store

## 16. Basic Retrieval

## 17. Cross-Paper Diverse Retrieval

Basic semantic retrieval is necessary, but for synthesis tasks we also want **paper diversity**. If the top chunks all come from the same paper, the final answer will sound confident but remain incomplete.

The diverse retriever below fetches a larger candidate set, then greedily selects chunks so that at least two papers are represented before filling the rest of the result list.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("multi_paper_research_qa")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="multi_paper_research_qa",
    metadata={"hnsw:space": "cosine"},
)

texts = [chunk["text"] for chunk in all_chunks]
metadatas = [chunk["metadata"] for chunk in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
vectors = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vectors, metadatas=metadatas, ids=ids)

print(f"Indexed chunks: {collection.count()}")


def retrieve(query: str, top_k: int = TOP_K, where: Optional[dict] = None) -> list[dict]:
    query_vector = embeddings.embed_query(query)
    kwargs = {"query_embeddings": [query_vector], "n_results": top_k}
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)
    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return chunks


def retrieve_diverse(query: str, top_k: int = TOP_K, min_papers: int = MIN_PAPERS) -> list[dict]:
    candidates = retrieve(query, top_k=top_k * 4)
    selected = []
    seen_papers = set()

    for chunk in candidates:
        paper_id = chunk["metadata"]["paper_id"]
        if len(seen_papers) < min_papers and paper_id not in seen_papers:
            selected.append(chunk)
            seen_papers.add(paper_id)
        if len(selected) == top_k:
            return selected

    selected_keys = {(c["metadata"]["paper_id"], c["metadata"]["section"], c["metadata"]["chunk_index"]) for c in selected}
    for chunk in candidates:
        key = (chunk["metadata"]["paper_id"], chunk["metadata"]["section"], chunk["metadata"]["chunk_index"])
        if key not in selected_keys:
            selected.append(chunk)
            selected_keys.add(key)
        if len(selected) == top_k:
            break

    return selected


def display_chunks(chunks: list[dict], max_chars: int = 180):
    for i, chunk in enumerate(chunks, 1):
        meta = chunk["metadata"]
        similarity = 1 - chunk["distance"]
        print(f"\n[{i}] sim={similarity:.3f} | {short_citation(meta)}")
        print(textwrap.shorten(chunk["text"].replace("\n", " "), width=max_chars, placeholder="..."))


query = "What improves evidence coverage for scientific synthesis questions?"
print("STANDARD RETRIEVAL")
standard = retrieve(query, top_k=4)
display_chunks(standard)
print(f"Unique papers: {len({c['metadata']['paper_id'] for c in standard})}")

print("\nDIVERSE RETRIEVAL")
diverse = retrieve_diverse(query, top_k=4)
display_chunks(diverse)
print(f"Unique papers: {len({c['metadata']['paper_id'] for c in diverse})}")

## 18. What Is Multi-Hop Retrieval?

Multi-hop retrieval is a retrieval strategy for questions that secretly contain multiple linked information needs.

Example question:

> *Which retrieval strategy improves evidence coverage, and what later verification step makes the final synthesis more trustworthy?*

This is not one lookup. It contains at least two hops:
1. find the retrieval strategy that improves evidence coverage
2. find the validation step that improves faithfulness or trust

A single similarity search may over-focus on only one part of the question. Multi-hop retrieval decomposes the question into smaller sub-questions, retrieves each one separately, then merges the evidence.

## 19. Build a Multi-Hop Planner

We use the local LLM as a planner. It returns a small JSON plan with 2 to 3 sub-questions.

In [ ]:
PLAN_SYSTEM = "You decompose research questions into precise retrieval hops. Keep them narrow and answerable from paper evidence."


def decompose_question(question: str) -> dict:
    prompt = (
        "Break this research question into 2 or 3 retrieval hops.\n\n"
        f"QUESTION: {question}\n\n"
        "Return ONLY JSON with keys: subquestions (list of strings), why_multi_hop."
    )
    response = ask(prompt, system=PLAN_SYSTEM, temperature=TEMP_PLAN)
    parsed = parse_json(response)
    if parsed and parsed.get("subquestions"):
        return parsed
    fallback = [part.strip().capitalize() for part in re.split(r"\band\b|\bthen\b", question) if part.strip()]
    return {
        "subquestions": fallback[:3] if fallback else [question],
        "why_multi_hop": "Fallback decomposition based on conjunction splitting.",
    }


def multi_hop_retrieve(question: str, hop_k: int = 3) -> dict:
    plan = decompose_question(question)
    merged = []
    seen = set()

    for hop_index, subquestion in enumerate(plan["subquestions"], start=1):
        hop_chunks = retrieve_diverse(subquestion, top_k=hop_k)
        for chunk in hop_chunks:
            meta = chunk["metadata"]
            key = (meta["paper_id"], meta["section"], meta["chunk_index"])
            if key not in seen:
                seen.add(key)
                merged.append({**chunk, "hop": hop_index, "subquestion": subquestion})

    return {
        "plan": plan,
        "chunks": merged,
    }


complex_question = "Which retrieval strategy improves evidence coverage, and what validation step makes the final answer more trustworthy?"
plan = decompose_question(complex_question)
print(json.dumps(plan, indent=2))

retrieval_plan = multi_hop_retrieve(complex_question)
print("\nMulti-hop evidence:")
for chunk in retrieval_plan["chunks"][:6]:
    print(f"Hop {chunk['hop']}: {chunk['subquestion']}")
    print(f"  -> {short_citation(chunk['metadata'])}")

## 20. Multi-Paper Research QA Prompt

The answer generator must do four things well:
1. synthesize across papers instead of picking one
2. cite the exact paper and section for each core claim
3. say whether the answer is supported by multiple papers
4. lower confidence if evidence is thin or single-source

## 21. Answer Research Questions

## 22. Ask a Cross-Paper Question

The first example is a synthesis question that should use at least two papers.

In [ ]:
ANSWER_SYSTEM = """You answer research questions using ONLY the supplied evidence passages.

Rules:
1. Prefer source-backed synthesis over generic explanation.
2. Cite claims inline as [Paper: title | section].
3. If the evidence supports multiple papers, explicitly say the conclusion is cross-paper.
4. If only one paper supports a claim, say that the evidence is single-source.
5. Never invent studies, metrics, or claims.
6. Keep the answer precise and comparative."""


def format_sources(chunks: list[dict]) -> str:
    formatted = []
    for i, chunk in enumerate(chunks, start=1):
        meta = chunk["metadata"]
        hop_prefix = ""
        if "hop" in chunk:
            hop_prefix = f"(Hop {chunk['hop']}: {chunk['subquestion']}) "
        formatted.append(
            f"[{i}] {hop_prefix}{meta['title']} ({meta['year']}) | {meta['section']}\n"
            f"Venue: {meta['venue']}\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(formatted)


def answer_research(question: str, strategy: str = "diverse") -> dict:
    if strategy == "multi_hop":
        retrieval = multi_hop_retrieve(question)
        chunks = retrieval["chunks"]
        plan = retrieval["plan"]
    elif strategy == "standard":
        chunks = retrieve(question, top_k=TOP_K)
        plan = None
    else:
        chunks = retrieve_diverse(question, top_k=TOP_K)
        plan = None

    prompt = (
        "Answer the research question below using ONLY the supplied paper evidence.\n\n"
        f"QUESTION: {question}\n\n"
        f"EVIDENCE:\n{format_sources(chunks)}\n\n"
        "Return ONLY JSON with keys: answer, papers_used, sources_used, used_multiple_papers, confidence, confidence_reason, evidence_summary."
    )

    response = ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)
    parsed = parse_json(response)
    if not parsed:
        parsed = {
            "answer": response,
            "papers_used": sorted({c['metadata']['title'] for c in chunks}),
            "sources_used": [short_citation(c['metadata']) for c in chunks],
            "used_multiple_papers": len({c['metadata']['paper_id'] for c in chunks}) > 1,
            "confidence": "unknown",
            "confidence_reason": "LLM did not return valid JSON.",
            "evidence_summary": [],
        }
    parsed["chunks"] = chunks
    parsed["plan"] = plan
    return parsed


def display_answer(question: str, result: dict):
    print(f"\n{'=' * 90}")
    print(f"Q: {question}")
    print(f"Confidence: {result.get('confidence', '?')}")
    print(f"Multiple papers used: {result.get('used_multiple_papers', False)}")
    print(f"{'=' * 90}\n")
    wrap_print(result.get("answer", ""))
    if result.get("plan"):
        print("\nMulti-hop plan:")
        for idx, subquestion in enumerate(result['plan'].get('subquestions', []), start=1):
            print(f"  {idx}. {subquestion}")
    print("\nPapers used:")
    for title in result.get("papers_used", []):
        print(f"  - {title}")


question_1 = "How do section-aware chunking and query decomposition complement each other in scientific QA?"
result_1 = answer_research(question_1, strategy="diverse")
display_answer(question_1, result_1)

## 23. Ask a Multi-Hop Question

This second example is intentionally harder. It asks for a retrieval method from one paper and a downstream validation step from another paper. That is exactly the kind of question where multi-hop retrieval helps.

## 24. Build an Evidence Matrix

An evidence matrix is a compact way to see which paper contributed which claim to the final answer.

## 25. Evaluate Retrieval Coverage

We care not only about semantic relevance, but also about whether the answer is supported by multiple distinct papers.

In [ ]:
question_2 = "Which retrieval strategy improves evidence coverage, and what later verification step makes the synthesized answer more trustworthy?"
result_2 = answer_research(question_2, strategy="multi_hop")
display_answer(question_2, result_2)

print("\nEVIDENCE MATRIX")
print("-" * 90)
for item in result_2.get("evidence_summary", []):
    print(f"- {item.get('paper', '')}: {item.get('claim', '')}")

print("\nTOP EVIDENCE CHUNKS")
for chunk in result_2["chunks"][:6]:
    hop_text = f"Hop {chunk['hop']}" if "hop" in chunk else "Hop ?"
    print(f"- {hop_text} | {short_citation(chunk['metadata'])}")

coverage_tests = [
    {
        "question": "How do section-aware chunking and query decomposition complement each other in scientific QA?",
        "expected_papers": {"p1", "p2"},
        "strategy": "diverse",
    },
    {
        "question": "Which retrieval strategy improves evidence coverage, and what later verification step makes the synthesized answer more trustworthy?",
        "expected_papers": {"p2", "p3"},
        "strategy": "multi_hop",
    },
    {
        "question": "Why does citation verification work better when retrieval already covers multiple papers?",
        "expected_papers": {"p2", "p3"},
        "strategy": "diverse",
    },
]

print("\nRETRIEVAL COVERAGE EVALUATION")
print("-" * 90)
for test in coverage_tests:
    answer = answer_research(test["question"], strategy=test["strategy"])
    found = {chunk["metadata"]["paper_id"] for chunk in answer["chunks"]}
    matched = test["expected_papers"].issubset(found)
    status = "OK" if matched else "MISS"
    print(f"{status:4s} | {test['strategy']:9s} | papers found={sorted(found)}")
    print(f"     question: {test['question']}")

## 26. Common Mistakes

| Mistake | Why It Fails | Better Approach |
|--------|---------------|-----------------|
| Retrieve only top chunks by similarity | Often collapses onto one paper | Add paper-diverse reranking |
| Ask one broad literature question in one shot | Linked sub-questions get blended | Decompose into multi-hop retrieval steps |
| Treat all chunks equally | Methods, results, and limitations play different roles | Preserve section metadata |
| Generate answers without citation rules | Fluent but weakly grounded synthesis | Force explicit inline paper citations |
| Use verification instead of retrieval quality | Verification can only inspect retrieved evidence | Fix retrieval coverage first |

## 27. Production Improvement Ideas

1. Replace the synthetic corpus with real PDFs or ArXiv XML metadata
2. Add BM25 or hybrid retrieval so exact terminology is less likely to be missed
3. Add cross-encoder reranking after the initial vector search
4. Store paper abstracts separately and use them as a paper-level routing signal
5. Add explicit contradiction detection for papers that disagree
6. Add citation span alignment so each sentence in the answer maps to a specific evidence passage

## 28. Exercises

1. Add a fourth paper about reranking and show when it improves relevance but hurts paper diversity
2. Change the diverse retriever so it guarantees one evidence chunk from each requested topic
3. Build a `compare_papers(question)` helper that returns a side-by-side matrix of claims and evidence

## 29. Mini Challenge

Build a mini literature-review assistant that answers: *"What combination of retrieval and verification choices would you deploy for a trustworthy scientific synthesis system, and why?"*

Your answer should name at least two retrieval-stage decisions and one post-generation verification decision, each with supporting citations.

In [ ]:
print("SESSION SUMMARY")
print("=" * 72)
print(f"Papers indexed: {len(PAPERS)}")
print(f"Sections parsed: {len(all_sections)}")
print(f"Chunks indexed: {len(all_chunks)}")
print("Core capabilities built:")
print("- multi-paper indexing with section metadata")
print("- basic and paper-diverse retrieval")
print("- multi-hop question decomposition")
print("- source-backed research QA with citations")
print("- coverage evaluation based on distinct papers")

## 30. Session Summary

This notebook built a multi-paper literature QA pipeline that:
- indexes multiple papers with section-aware metadata
- retrieves evidence from more than one paper when the question requires synthesis
- explains and implements multi-hop retrieval for linked sub-questions
- produces source-backed answers with paper-and-section citations

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Multi-paper QA is a coverage problem as much as a ranking problem** |
| 2 | **Section-aware chunking improves retrieval precision for scientific text** |
| 3 | **Paper diversity matters** because one highly similar paper can crowd out the rest |
| 4 | **Multi-hop retrieval** is useful when a question contains multiple linked information needs |
| 5 | **Verification is downstream insurance, not a substitute for good retrieval** |
| 6 | **Cross-paper synthesis should say when it is truly multi-source and when it is not** |

---

*This notebook follows the same local, Ollama-first teaching style used across the NLP notebooks in this repository.*

*Replace the synthetic sample corpus with parsed PDFs or ArXiv text when you want to turn this into a real literature assistant.*